# This is the header
## This is a subheader
### This is the content of the topic.

* Markdown or .md format is demonstrated

In [1]:
import os
import json
import pandas as pd
from dotenv import load_dotenv

# load .env
load_dotenv()

# load the config
with open("config/config.json", "r") as f:
    config = json.load(f)

print(config["llm"])

{'openai_model': 'gpt-4o-mini', 'gemini_model': 'gemini-1.5-flash', 'temperature': 0.3}


In [2]:
df = pd.read_csv("data/transcripts.csv")

print("Data Loaded")
df.head()

Data Loaded


,call_id,agent_name,transcript,expected_call_type
0,1,Alice,Customer: I was charged twice for my premium t...,billing
1,2,Bob,Customer: My claim has been pending for 2 week...,claims
2,3,Charlie,Customer: This is the third time I'm calling! ...,complaint
3,4,Diana,Customer: Can you explain my coverage details?...,general_query


In [3]:
df.tail(2)

,call_id,agent_name,transcript,expected_call_type
2,3,Charlie,Customer: This is the third time I'm calling! ...,complaint
3,4,Diana,Customer: Can you explain my coverage details?...,general_query


In [4]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

def load_llm(config):
    provider = os.getenv("LLM_PROVIDER")

    if provider == "openai":
        llm = ChatOpenAI(
            model = config["llm"]["openai_model"],
            temperature = config["llm"]["temperature"]
        )
    elif provider == "gemini":
        llm = ChatGoogleGenerativeAI(
            model = config["llm"]["gemini_model"],
            temperature = config["llm"]["temperature"]
        )
    else:
        raise ValueError("Invalid LLM_PROVIDER")
    return llm

llm = load_llm(config)
print(f"LLM initialized using provider: {os.getenv('LLM_PROVIDER')}")

LLM initialized using provider: openai


In [5]:
response = llm.invoke("Say 'setup successful' in one short sentence")

print(f"LLM Response: {response.content}")

LLM Response: Setup successful.


In [6]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field


# Define output schema in a class
class ClassificationOutput(BaseModel):
    call_type: str = Field(description = "Type of customer call")
    confidence: float = Field(description = "Confidence score between 0 and 1")

parser = PydanticOutputParser(pydantic_object = ClassificationOutput)


# Prompt template
prompt = PromptTemplate(
    template="""
You are a call classification assistant.

Classify the following customer support transcript into one of these categories:
{labels}

Transcript:
{transcript}

{format_instructions}
""",
    input_variables=["transcript"],
    partial_variables={
        "format_instructions": parser.get_format_instructions(),
        "labels": config["classification"]["labels"]
    }
)


# create chain
classification_chain = prompt | llm | parser

# test chain with one transcript
sample_text = df.iloc[0]["transcript"]

result = classification_chain.invoke({
    "transcript": sample_text
})

print(f"Classification Result: \n{result}")

Classification Result: 
call_type='billing' confidence=0.95


In [7]:
from tqdm import tqdm

results = []

for i, row in tqdm(df.iterrows(), total=len(df), desc = "Classifying Calls"):

    try:
        output = classification_chain.invoke({
            "transcript": row["transcript"]
        })

        results.append({
            "call_id": row["call_id"],
            "predicted_call_type": output.call_type,
            "confidence": output.confidence
        })
    except Exception as e:
        print(f"Error at row -> {i}: {e}")
        results.append({
            "call_id": row["call_id"],
            "predicted_call_type": None,
            "confidence": None
        })


results_df = pd.DataFrame(results)
df = df.merge(results_df, on="call_id")

df.head()

Classifying Calls: 100%|██████████| 4/4 [00:04<00:00,  1.20s/it]


,call_id,agent_name,transcript,expected_call_type,predicted_call_type,confidence
0,1,Alice,Customer: I was charged twice for my premium t...,billing,billing,0.95
1,2,Bob,Customer: My claim has been pending for 2 week...,claims,claims,0.95
2,3,Charlie,Customer: This is the third time I'm calling! ...,complaint,complaint,0.90
3,4,Diana,Customer: Can you explain my coverage details?...,general_query,general_query,0.90


In [8]:
# --- Step 1: Define routing logic ---

def route_call(call_type):
    if call_type == "billing":
        return ["knowledge_accuracy", "resolution_quality"]
    
    elif call_type == "claims":
        return ["knowledge_accuracy", "resolution_quality"]
    
    elif call_type == "complaint":
        return ["tone_empathy", "resolution_quality"]
    
    elif call_type == "general_query":
        return ["knowledge_accuracy"]
    
    else:
        return ["knowledge_accuracy"]  # fallback


# --- Step 2: Apply routing to dataset ---

df["evaluation_plan"] = df["predicted_call_type"].apply(route_call)

print("✅ Routing Applied\n")
print(df[["call_id", "predicted_call_type", "evaluation_plan"]])

✅ Routing Applied

   call_id predicted_call_type                           evaluation_plan
0        1             billing  [knowledge_accuracy, resolution_quality]
1        2              claims  [knowledge_accuracy, resolution_quality]
2        3           complaint        [tone_empathy, resolution_quality]
3        4       general_query                      [knowledge_accuracy]


In [9]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# define an output schema for tone evaluation
class ToneEvaluation(BaseModel):
    score: int = Field(description = "Score between 1 and 5")
    reasoning: str = Field(description = "Explanation of the score")

tone_parser = PydanticOutputParser(pydantic_object = ToneEvaluation)

tone_prompt = PromptTemplate(
    template = """
    You are a QA evaluator for customer support calls.

    Evaluate the agent's tone and empathy in the following transcript.

    Consider:
    - Did the agent acknowledge the cusotmer's issue?
    - Was the tone polite and professional enough?
    - Did the agent show empathy?

    Transcript:
    {transcript}

    {format_instructions}
    """,
    input_variables = ["transcript"],
    partial_variables = {
        "format_instructions": tone_parser.get_format_instructions()
    }

)

tone_chain = tone_prompt | llm | tone_parser

sample_text = df[df["predicted_call_type"] == "complaint"].iloc[0]["transcript"]

result = tone_chain.invoke({
    "transcript": sample_text
})

print(f"Tone evaluation result: \n{result}")

Tone evaluation result: 
score=3 reasoning="The agent acknowledged the customer's frustration by apologizing for the inconvenience, which shows some level of empathy. However, the response was quite brief and did not fully address the customer's repeated issues or express a deeper understanding of their situation. The tone was polite, but it lacked warmth and reassurance that the agent would actively work to resolve the problem."


In [10]:
# --- Step 1: Schema ---
class ResolutionEvaluation(BaseModel):
    score: int = Field(description="Score between 1 and 5")
    reasoning: str = Field(description="Explanation of the score")

resolution_parser = PydanticOutputParser(pydantic_object=ResolutionEvaluation)

# --- Step 2: Prompt ---
resolution_prompt = PromptTemplate(
    template="""
You are a QA evaluator for customer support calls.

Evaluate the resolution quality of the agent.

Consider:
- Did the agent fully resolve the customer's issue?
- Were next steps clearly communicated?
- Did the agent confirm resolution before ending?

Transcript:
{transcript}

{format_instructions}
""",
    input_variables=["transcript"],
    partial_variables={
        "format_instructions": resolution_parser.get_format_instructions()
    }
)

# --- Step 3: Chain ---
resolution_chain = resolution_prompt | llm | resolution_parser

# --- Step 4: Test
sample_text = df.iloc[0]["transcript"]

result = resolution_chain.invoke({
    "transcript": sample_text
})

print("✅ Resolution Evaluation Result:")
print(result)

✅ Resolution Evaluation Result:
score=2 reasoning="The agent did not fully resolve the customer's issue as the transcript does not show any follow-up actions taken to address the double charge. There are no next steps communicated, and the agent did not confirm resolution before ending the call."


In [11]:
# --- Step 1: Imports (safe to repeat) ---
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# --- Step 2: Schema ---
class KnowledgeEvaluation(BaseModel):
    score: int = Field(description="Score between 1 and 5")
    reasoning: str = Field(description="Explanation of the score")

knowledge_parser = PydanticOutputParser(pydantic_object=KnowledgeEvaluation)

# --- Step 3: Prompt ---
knowledge_prompt = PromptTemplate(
    template="""
You are a QA evaluator for customer support calls.

Evaluate the agent's knowledge accuracy and clarity.

Consider:
- Did the agent provide correct and relevant information?
- Was the explanation clear and easy to understand?
- Did the agent avoid vague or misleading statements?

IMPORTANT:
- If the transcript does not contain enough information, give a moderate score (2 or 3) and explain why.

Transcript:
{transcript}

{format_instructions}
""",
    input_variables=["transcript"],
    partial_variables={
        "format_instructions": knowledge_parser.get_format_instructions()
    }
)

# --- Step 4: Chain ---
knowledge_chain = knowledge_prompt | llm | knowledge_parser

print("✅ Knowledge Chain Ready")

✅ Knowledge Chain Ready


In [12]:
from tqdm import tqdm

def run_evaluations(transcript, eval_plan):
    results = {}

    if "tone_empathy" in eval_plan:
        try:
            tone_result = tone_chain.invoke({"transcript": transcript})
            results["tone"] = tone_result.model_dump()
        except Exception as e:
            results["tone"] = {"error": str(e)}

    if "knowledge_accuracy" in eval_plan:
        try:
            knowledge_result = knowledge_chain.invoke({"transcript": transcript})
            results["knowledge"] = knowledge_result.model_dump()
        except Exception as e:
            results["knowledge"] = {"error": str(e)}

    if "resolution_empathy" in eval_plan:
        try:
            resolution_result = resolution_chain.invoke({"transcript": transcript})
            results["resolution"] = resolution_result.model_dump()
        except Exception as e:
            results["resolution"] = {"error": str(e)}

    return results


evaluation_outputs = []

for i, row in tqdm(df.iterrows(), total=len(df), desc="Running Evaluations"):
    output = run_evaluations(row["transcript"], row["evaluation_plan"])
    
    evaluation_outputs.append({
        "call_id": row["call_id"],
        "evaluation_output": output
    })

# Convert to DataFrame
eval_df = pd.DataFrame(evaluation_outputs)

# Merge
df = df.merge(eval_df, on="call_id")

print("\n✅ Evaluation Completed\n")

# Show one example clearly
import pprint
pprint.pprint(df.iloc[0]["evaluation_output"])



Running Evaluations: 100%|██████████| 4/4 [00:05<00:00,  1.31s/it]


✅ Evaluation Completed

{'knowledge': {'reasoning': "The agent's response to the customer's issue "
                            'about being charged twice is incomplete. While '
                            'the agent indicates they will check the issue, '
                            'there is no information provided about the '
                            "customer's concern or any relevant details "
                            'regarding the charge. Therefore, the knowledge '
                            'accuracy cannot be fully assessed, leading to a '
                            'moderate score.',
               'score': 3}}


In [ ]:
# --- Step 1: Schema ---
class FinalReport(BaseModel):
    summary: str = Field(description="Overall evaluation summary")
    recommendations: list[str] = Field(description="List of actionable improvements")

final_parser = PydanticOutputParser(pydantic_object=FinalReport)

# --- Step 2: Prompt ---
final_prompt = PromptTemplate(
    template="""
You are a QA manager reviewing customer support calls.

Based on the evaluation results below, generate:

1. A concise summary of the agent's performance
2. A list of actionable recommendations for improvement

Evaluation Data:
{evaluation_output}

IMPORTANT:
- Be specific and practical
- Do not repeat scores
- Focus on improvement

{format_instructions}
""",
    input_variables=["evaluation_output"],
    partial_variables={
        "format_instructions": final_parser.get_format_instructions()
    }
)

# --- Step 3: Chain ---
final_chain = final_prompt | llm | final_parser

# --- Step 4: Test on one row
sample_eval = df.iloc[0]["evaluation_output"]

result = final_chain.invoke({
    "evaluation_output": sample_eval
})

print("✅ Final QA Report:")
print(result)